# Notes for 3/6/26 Class #

This class is about working with LLMs using Ollama and related software.  If you are reading this on the Github, note that the work in this notebook was done on the external GPU servers, not euclid.

Documentation for Ollama can be found [here](https://github.com/ollama/ollama?tab=readme-ov-file).

## Ollama Python API overview ##

The goal in these next few cells is to show you how interaction with an LLM ('qwen3' in this case) takes place.  

The first cell shows how you make a call to an LLM.  **Note:** The first call will generally always be slow since Ollama has to load the model into GPU memory, which typically takes some time. 

The cell below givees **two** versions of the same response, showing the different ways the LLM response can be accessed. 

Note also that the response is in Markdown format.

In [25]:
from ollama import chat
from ollama import ChatResponse

In [27]:
response: ChatResponse = chat(model='qwen3', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky appears blue due to a phenomenon called **Rayleigh scattering**, which involves how sunlight interacts with Earth's atmosphere. Here's a breakdown of the key reasons:

1. **Sunlight and the Spectrum**:  
   Sunlight is white but consists of a mix of all visible colors (red, orange, yellow, green, blue, indigo, violet). Each color has a different wavelength, with **blue and violet** having the shortest wavelengths.

2. **Scattering in the Atmosphere**:  
   When sunlight enters Earth's atmosphere, it collides with molecules (like nitrogen and oxygen) and small particles. These interactions **scatter light in all directions**, but **shorter wavelengths (blue/violet) scatter more** than longer ones (red/orange). This is called **Rayleigh scattering**.

3. **Why Blue, Not Violet?**  
   - The **sun emits more blue light** than violet.  
   - Human eyes are **more sensitive to blue light** than violet.  
   - Some **violet light is absorbed by the upper atmosphere**, reducing its vi

## The Response Structure ##

Note that the actual response you see on the screen is a value corresponding to a key ("content") that is itself a value in the content of another key ("message").  The overall response structure is essentially a json object.

The next few cells look at this response and attempt (not very successfully) to "pretty print" the JSON object given by the response.

In [13]:
response

ChatResponse(model='qwen3', created_at='2026-03-06T14:54:48.181902255Z', done=True, done_reason='stop', total_duration=12060542090, load_duration=2673907492, prompt_eval_count=16, prompt_eval_duration=144054402, eval_count=796, eval_duration=7511569830, message=Message(role='assistant', content="The sky appears blue due to a phenomenon called **Rayleigh scattering**, which involves how sunlight interacts with the Earth's atmosphere. Here's a simplified breakdown:\n\n1. **Sunlight Composition**: Sunlight is a mix of all visible colors (red, orange, yellow, green, blue, indigo, violet). These colors combine to form white light.\n\n2. **Atmospheric Particles**: The Earth's atmosphere contains molecules like nitrogen and oxygen, which are much smaller than the wavelengths of visible light. When sunlight enters the atmosphere, these molecules scatter the light in all directions.\n\n3. **Scattering Effect**: Shorter wavelengths of light (blue and violet) scatter more easily than longer wavel

### It may be easier to understand this with a simpler example.  ##

In [28]:
# Generate response
response2 = chat(model='qwen3', messages=[{'role': 'user', 'content': 'Hello'}])

# Access the generated text
print(response2['message']['content'])

Hello! How can I assist you today? 😊


In [29]:
response2

ChatResponse(model='qwen3', created_at='2026-03-07T23:12:13.062431654Z', done=True, done_reason='stop', total_duration=3761445793, load_duration=154656535, prompt_eval_count=11, prompt_eval_duration=2231878381, eval_count=122, eval_duration=1121410940, message=Message(role='assistant', content='Hello! How can I assist you today? 😊', thinking='Okay, the user said "Hello /think". I need to respond appropriately. First, I should acknowledge their greeting. Since they included "/think", maybe they\'re testing if I can process that command. I should check if there\'s any specific action needed with that. But generally, I should respond in a friendly manner. Let me make sure to keep the tone positive and open for further interaction. Also, I should avoid any markdown and keep the response natural. Alright, I\'ll start with a simple greeting and offer assistance.\n', images=None, tool_name=None, tool_calls=None), logprobs=None)

### Here is me trying to make this response easier to understand by dumping it to JSON and 'pretty printing' it.  Note the extra content in the response that you never see. ###

In [30]:
import json

In [31]:
response2.model_dump_json()

'{"model":"qwen3","created_at":"2026-03-07T23:12:13.062431654Z","done":true,"done_reason":"stop","total_duration":3761445793,"load_duration":154656535,"prompt_eval_count":11,"prompt_eval_duration":2231878381,"eval_count":122,"eval_duration":1121410940,"message":{"role":"assistant","content":"Hello! How can I assist you today? 😊","thinking":"Okay, the user said \\"Hello /think\\". I need to respond appropriately. First, I should acknowledge their greeting. Since they included \\"/think\\", maybe they\'re testing if I can process that command. I should check if there\'s any specific action needed with that. But generally, I should respond in a friendly manner. Let me make sure to keep the tone positive and open for further interaction. Also, I should avoid any markdown and keep the response natural. Alright, I\'ll start with a simple greeting and offer assistance.\\n","images":null,"tool_name":null,"tool_calls":null},"logprobs":null}'

In [32]:
jsonresponse2 = response2.model_dump_json()

In [33]:
pretty_json = json.dumps(jsonresponse2, indent=4)
print(pretty_json)

"{\"model\":\"qwen3\",\"created_at\":\"2026-03-07T23:12:13.062431654Z\",\"done\":true,\"done_reason\":\"stop\",\"total_duration\":3761445793,\"load_duration\":154656535,\"prompt_eval_count\":11,\"prompt_eval_duration\":2231878381,\"eval_count\":122,\"eval_duration\":1121410940,\"message\":{\"role\":\"assistant\",\"content\":\"Hello! How can I assist you today? \ud83d\ude0a\",\"thinking\":\"Okay, the user said \\\"Hello /think\\\". I need to respond appropriately. First, I should acknowledge their greeting. Since they included \\\"/think\\\", maybe they're testing if I can process that command. I should check if there's any specific action needed with that. But generally, I should respond in a friendly manner. Let me make sure to keep the tone positive and open for further interaction. Also, I should avoid any markdown and keep the response natural. Alright, I'll start with a simple greeting and offer assistance.\\n\",\"images\":null,\"tool_name\":null,\"tool_calls\":null},\"logprobs\":

In [34]:
import pprint

In [35]:
pprint.pprint(jsonresponse2)

('{"model":"qwen3","created_at":"2026-03-07T23:12:13.062431654Z","done":true,"done_reason":"stop","total_duration":3761445793,"load_duration":154656535,"prompt_eval_count":11,"prompt_eval_duration":2231878381,"eval_count":122,"eval_duration":1121410940,"message":{"role":"assistant","content":"Hello! '
 'How can I assist you today? 😊","thinking":"Okay, the user said \\"Hello '
 '/think\\". I need to respond appropriately. First, I should acknowledge '
 'their greeting. Since they included \\"/think\\", maybe they\'re testing if '
 "I can process that command. I should check if there's any specific action "
 'needed with that. But generally, I should respond in a friendly manner. Let '
 'me make sure to keep the tone positive and open for further interaction. '
 'Also, I should avoid any markdown and keep the response natural. Alright, '
 "I'll start with a simple greeting and offer "
 'assistance.\\n","images":null,"tool_name":null,"tool_calls":null},"logprobs":null}')


# Langchain #

Ollama is software for accessing a library of LLMs that you have stored locally, and Langchain is a library for coordinating information flow to an LLM, using tools such as RAG and other LLMs.

See [this link](https://docs.langchain.com/oss/python/integrations/chat/ollama) for an overview of using ollama and Langchain.

The point here is to see an example of how to query an LLM with the Langchain API and print its response. 

In [42]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0,
    # other params...
)

In [43]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='The translation of "I love programming" to French is:\n\n"J\'adore le programmation."', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-03-07T23:16:35.390283702Z', 'done': True, 'done_reason': 'stop', 'total_duration': 372805959, 'load_duration': 161567360, 'prompt_eval_count': 35, 'prompt_eval_duration': 29616105, 'eval_count': 22, 'eval_duration': 173462332, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019cca96-a048-75f1-9791-9080af872351-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 22, 'total_tokens': 57})

In [44]:
print(ai_msg.content)

The translation of "I love programming" to French is:

"J'adore le programmation."
